# Junior rangemelő mérés – 3. blokk, 2. óra

**Egyéni munka, 40 perc.** Használhatod a saját jegyzeteidet és a `modell.ipynb`-t; MI-asszisztenst és mások segítségét nem.
Pontozás: F1–F4 feladatonként 2 pont (a CI ellenőrzi), F5 2 pont (a tanár értékeli). **A Junior ranghoz legalább 6 pont kell a 10-ből.**

**Beadás:** File → Save a copy in GitHub → saját repó, `main`, fájlnév `meres_junior.ipynb`, commitüzenet: `meres: Junior rangemelő mérés`. Utána a repó linkjét a Teams-feladat „Junior mérés” részébe add be.

In [1]:
import json
from pathlib import Path

import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import accuracy_score, mean_absolute_error, recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

ADAT = Path("data/hivasok_november.csv")
df = pd.read_csv(ADAT if ADAT.exists() else "https://raw.githubusercontent.com/Kizsi2025/bolyai-labs-03-modell/main/data/hivasok_november.csv")
df["modell_nagyobb"] = (df["modell"] == "nagyobb").astype(int)
print(df.shape)
print("Sikeres hívások aránya:", round(df["sikeres"].mean(), 3))
df.head()

(1000, 6)
Sikeres hívások aránya: 0.711


,hivas_id,modell,tokenek,valaszido_ms,sikeres,modell_nagyobb
0,1,nagyobb,949,826,1,1
1,2,nagyobb,908,925,0,1
2,3,nagyobb,689,826,0,1
3,4,nagyobb,1101,1090,0,1
4,5,nagyobb,764,844,0,1


In [4]:
X = df[["tokenek", "valaszido_ms", "modell_nagyobb"]]
y = df["sikeres"]

## F1 (2 pont) – Felosztás
Oszd fel az adatot: a teszthalmaz **30%** legyen, `random_state=7`, rétegzetten (`stratify=y`).

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=7, stratify=y)  # TODO
alap = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
baseline = accuracy_score(y_test, alap.predict(X_test))
print(len(X_test), round(baseline, 3))

300 0.71


## F2 (2 pont) – Döntési fa
Taníts egy `DecisionTreeClassifier(max_depth=4, random_state=0)` modellt. Számold ki a pontosságot és a sikertelen osztály felidézését.

In [14]:
fa = DecisionTreeClassifier(max_depth=4, random_state=0).fit(X_train, y_train)  # TODO
f2_accuracy = accuracy_score(fa.predict(X_test), y_test)  # TODO
f2_recall_hiba = recall_score(fa.predict(X_test), y_test, pos_label=0)  # TODO
print(round(f2_accuracy, 3), round(f2_recall_hiba, 3))

0.753 0.576


## F3 (2 pont) – Kapd el a hibákat!
Készíts egy modellt, amely a sikertelen hívások **legalább 80%-át** megtalálja (felidézés ≥ 0,80), és közben a pontossága **legalább 0,72**. Bármilyen tanult modellt és beállítást használhatsz (tipp: `class_weight`).

In [22]:
sajat = make_pipeline(StandardScaler(), LogisticRegression()).fit(X_train, y_train) # TODO
f3_accuracy = accuracy_score(sajat.predict(X_test), y_test)  # TODO
f3_recall_hiba = recall_score(sajat.predict(X_test), y_test, pos_label=0)  # TODO
print(round(f3_accuracy, 3), round(f3_recall_hiba, 3))

0.793 0.624


## F4 (2 pont) – Regresszió
Jósold meg a válaszidőt (`valaszido_ms`) bármilyen jellemzőkkel, a sikeres oszlop kivételével. A felosztás: `test_size=0.3, random_state=7`. A teszthalmazon az átlagos abszolút hiba **60 ms alatt** legyen.

In [ ]:
f4_mae = ...  # TODO
print(round(f4_mae, 1))

## F5 (2 pont) – Értékelés (a tanár pontozza)

Írj ide 3–4 mondatot: melyik modellt ajánlanád a Bolyai Labsnak, ha a cél az, hogy a sikertelen hívásokat időben átirányítsuk? Miért nem elég ehhez a pontosságot nézni? Mi az ára annak, ha sok hibát akarunk elkapni?

*Ide írd a válaszod.*

## Mentés – ezt a cellát ne módosítsd

In [ ]:
Path("eredmenyek").mkdir(exist_ok=True)
json.dump({"f1_teszt_meret": len(X_test), "f1_baseline": baseline,
           "f2_accuracy": f2_accuracy, "f2_recall_hiba": f2_recall_hiba,
           "f3_accuracy": f3_accuracy, "f3_recall_hiba": f3_recall_hiba, "f4_mae": f4_mae},
          open("eredmenyek/meres.json", "w"), indent=1)
print("Mentve.")